In [1]:
!pip install trl transformers datasets accelerate sympy math_verify pylatexenc

In [2]:
import os
import torch
import numpy as np
import random
import pandas as pd
from typing import Dict, List, Optional, Callable
from dataclasses import dataclass
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    TrainerCallback
)
from trl import GRPOTrainer, GRPOConfig, ModelConfig
from trl.trainer.utils import disable_dropout_in_model
from drgrpo_grader import r1_zero_reward_fn

/opt/conda/envs/py_3.12/lib/python3.12/site-packages/redis/connection.py:77: UserWarning: redis-py works best with hiredis. Please consider installing
  warnings.warn(msg)


In [3]:
@dataclass
class ScriptConfig:
    """Configuration for the GRPO training script."""
    # Model and data
    model_name: str = 'Qwen/Qwen2.5-Math-1.5B'
    train_dataset_path: str = "math_12k_train.parquet"
    test_dataset_path: str = "math_12k_test.parquet"

    # Training hyperparameters
    learning_rate: float = 1e-5
    num_train_epochs: float = 1.0
    per_device_train_batch_size: int = 2  # Adjusted for memory
    per_device_eval_batch_size: int = 32
    gradient_accumulation_steps: int = 16  # To achieve effective batch size
    warmup_steps: int = 10
    logging_steps: int = 1
    eval_steps: int = 10
    save_steps: int = 50

    # GRPO specific
    group_size: int = 8
    advantage_eps: float = 1e-6
    loss_type: str = "reinforce_with_baseline"
    normalize_advantages: bool = True

    # Generation parameters
    sampling_temperature: float = 1.0
    max_new_tokens: int = 1024
    min_new_tokens: int = 4

    # System
    seed: int = 42
    bf16: bool = True
    gradient_checkpointing: bool = True
    dataloader_num_workers: int = 4
    remove_unused_columns: bool = False

In [4]:
def set_seed(seed: int):
    """Set random seeds for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)

def format_prompt(question: str) -> str:
    """Format the input question into the required prompt format."""
    return f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>"""

def load_dataset(file_path: str) -> Dataset:
    """Load and format dataset for GRPO training."""
    df = pd.read_parquet(file_path)

    # Format the data for TRL
    formatted_data = []
    for _, row in df.iterrows():
        formatted_data.append({
            'prompt': format_prompt(row['problem']),
            'ground_truth': row['solution'],
            'problem': row['problem']
        })

    return Dataset.from_list(formatted_data)

In [5]:
class CustomRewardFunction:
    """Custom reward function wrapper for GRPO."""

    def __init__(self, reward_fn: Callable):
        self.reward_fn = reward_fn
        self.__name__ = 'custom_reward_function'

    def __call__(self, prompts, completions, completion_ids, **reward_kwargs) -> List[float]:
        """
        Compute rewards for a batch of responses.

        Args:
            responses: List of generated responses
            ground_truths: List of ground truth solutions

        Returns:
            List of reward values
        """
        # print(f"{prompts=}, {completions=}, {completion_ids=}")
        # print(reward_kwargs)
        ground_truths = reward_kwargs["ground_truth"]
        # print(f"{completions[0]}")
        # print(f"{ground_truths[0]}")
        rewards = []
        for response, truth in zip(completions, ground_truths):
            reward_dict = self.reward_fn(response, truth)
            # Use partial reward: 1.0 for correct answer, 0.2 for correct format, 0.0 otherwise
            if reward_dict['reward'] == 1.0:
                rewards.append(1.0)
            elif reward_dict['format_reward'] == 1.0:
                rewards.append(0.0)
            else:
                rewards.append(0.0)
        return rewards
        # return [0] * len(prompts)


In [6]:
class CustomGRPOTrainer(GRPOTrainer):
    """Extended GRPOTrainer with custom evaluation."""

    def __init__(self, *args, test_dataset=None, **kwargs):
        super().__init__(**kwargs)
        self.test_dataset = test_dataset

    def evaluate_model(self):
        """Evaluate model on test set."""
        if self.test_dataset is None:
            return

        print("Evaluating model...")
        self.model.eval()

        total_format_reward = 0
        total_answer_reward = 0
        total_reward = 0

        # Sample a subset for evaluation to save time
        eval_samples = self.test_dataset.shuffle(seed=42).select(range(min(1000, len(self.test_dataset))))

        for sample in eval_samples:
            # Generate response
            inputs = self.tokenizer.encode(sample['prompt'], return_tensors='pt').to(self.model.device)

            with torch.no_grad():
                outputs = self.model.generate(
                    inputs,
                    max_new_tokens=1024,
                    temperature=1.0,
                    do_sample=True,
                    pad_token_id=self.tokenizer.eos_token_id,
                    stop_strings=["</answer>"],
                )

            # Decode response
            response = self.tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)

            # Compute reward
            reward_dict = r1_zero_reward_fn(response, sample['ground_truth'])

            if reward_dict['format_reward'] == 1.0:
                total_format_reward += 1
            if reward_dict['answer_reward'] == 1.0:
                total_answer_reward += 1
            if reward_dict['reward'] == 1.0:
                total_reward += 1

        print(f"Format accuracy: {total_format_reward / len(eval_samples):.3f}")
        print(f"Answer accuracy: {total_answer_reward / len(eval_samples):.3f}")
        print(f"Total accuracy: {total_reward / len(eval_samples):.3f}")

In [7]:
config = ScriptConfig()
set_seed(config.seed)

# Load datasets
print("Loading datasets...")
train_dataset = load_dataset(config.train_dataset_path)
test_dataset = load_dataset(config.test_dataset_path)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# Load model and tokenizer
print("Loading model and tokenizer...")
model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    torch_dtype=torch.bfloat16 if config.bf16 else torch.float32,
    # attn_implementation='flash_attention_2',
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(config.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Disable dropout for more stable training
disable_dropout_in_model(model)

# Setup reward function
reward_function = CustomRewardFunction(r1_zero_reward_fn)

Loading datasets...


`torch_dtype` is deprecated! Use `dtype` instead!


Train dataset size: 7500
Test dataset size: 5000
Loading model and tokenizer...


In [8]:
 # Configure training arguments
training_args = GRPOConfig(
    output_dir="./grpo_math_training",
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=config.warmup_steps,
    logging_steps=config.logging_steps,
    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    bf16=config.bf16,
    gradient_checkpointing=config.gradient_checkpointing,
    dataloader_num_workers=config.dataloader_num_workers,
    remove_unused_columns=config.remove_unused_columns,
    report_to=None,  # Disable wandb/tensorboard
    save_strategy="no",
    eval_strategy="steps",
    load_best_model_at_end=False,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    generation_kwargs={
        "temperature": config.sampling_temperature,
        "max_new_tokens": config.max_new_tokens,
        "min_new_tokens": config.min_new_tokens,
        "do_sample": True,
        "pad_token_id": tokenizer.eos_token_id,
        # "stop_strings": ["</answer>"],  # FIXME ugly hack
        # "tokenizer": config.model_name,
    },
    model_init_kwargs={},
)


In [9]:
# Initialize trainer
trainer = CustomGRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_processing_classes=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=reward_function,
    test_dataset=test_dataset,
    eval_dataset=test_dataset,
)

# Custom callback to run evaluation
class EvalCallback(TrainerCallback):
    def __init__(self, trainer):
        self.trainer = trainer

    def on_evaluate(self, args, state, control, **kwargs):
        if state.global_step % (args.eval_steps) == 0 and state.global_step > 0:
            self.trainer.evaluate_model()

trainer.add_callback(EvalCallback(trainer))

You passed `model_init_kwargs` to the `GRPOConfig`, but your model is already instantiated. The `model_init_kwargs` will be ignored.


ValueError: You have set `args.eval_strategy` to IntervalStrategy.STEPS but you didn't pass an `eval_dataset` to `Trainer`. Either set `args.eval_strategy` to `no` or pass an `eval_dataset`. 

In [ ]:
from accelerate import Accelerator

# Reinitialize Accelerator to ensure AcceleratorState is properly set up
accelerator = Accelerator()

In [ ]:
# Train
print("Starting training...")
trainer.train()

# Final evaluation
print("Final evaluation:")
trainer.evaluate_model()

# Save final model
trainer.save_model("./grpo_math_final")
print("Training completed!")